<div style="background:linear-gradient(135deg,#7a3d00 0%,#b45309 55%,#d97706 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#ffe9c7;font-weight:700;text-transform:uppercase">Chapter 24 · Solutions</div>
  <div style="font-size:36px;font-weight:900;line-height:1.1;margin:10px 0 6px">Practice Challenges, Worked Answers ✅</div>
  <div style="font-size:15px;color:#ffe6cc;max-width:700px;line-height:1.6">Full solutions to the five "Feature Engineering" challenges. Try them yourself first, then compare.</div>
  <div style="margin-top:16px;font-size:13px;color:#ffe2bf">Statistics, Data Science and AI: A Visual Handbook · John Fisher · 2026</div>
</div>

## ⚙️ Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(242)
print("Ready.")

Ready.


<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#0891b2;letter-spacing:1px">CHALLENGE 1 · ONE-HOT A CATEGORY</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Nominal to numbers</div>
<div style="color:#4a5578;margin-top:6px">One-hot encode a 'color' column with get_dummies. How many columns result, and why is one-hot (not integer labels) the right choice for a nominal feature in a linear model?</div>
</div>

In [2]:
df = pd.DataFrame({"color": ["red","green","blue","red","green"]})
oh = pd.get_dummies(df["color"], prefix="color").astype(int)
print(oh)
print(f"\n{df['color'].nunique()} categories -> {oh.shape[1]} one-hot columns")

   color_blue  color_green  color_red
0           0            0          1
1           0            1          0
2           1            0          0
3           0            0          1
4           0            1          0

3 categories -> 3 one-hot columns


**Answer:** Three categories become **three 0/1 columns** (`color_blue`, `color_green`, `color_red`). One-hot is correct because color is **nominal**, it has no order. Integer labels (red=0, green=1, blue=2) would tell a linear or distance model that blue is "twice" green and farther from red, a fake ordering. For an unregularized linear model you would also drop one column (`drop_first`) to avoid the dummy-variable trap.

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#7c3aed;letter-spacing:1px">CHALLENGE 2 · ENCODE AN ORDINAL</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Keep the real order</div>
<div style="color:#4a5578;margin-top:6px">A 'rating' column has values Poor, Fair, Good, Excellent. Encode it so the order is preserved, and explain why label-encoding it in alphabetical order would be wrong.</div>
</div>

In [3]:
df = pd.DataFrame({"rating": ["Good","Poor","Excellent","Fair","Good"]})
order = ["Poor","Fair","Good","Excellent"]
oe = OrdinalEncoder(categories=[order])
df["code"] = oe.fit_transform(df[["rating"]]).astype(int)
print(df)

      rating  code
0       Good     2
1       Poor     0
2  Excellent     3
3       Fair     1
4       Good     2


**Answer:** Give `OrdinalEncoder` the **explicit order** `["Poor","Fair","Good","Excellent"]` so it maps Poor=0 ... Excellent=3, preserving the real ranking. Letting an encoder assign codes **alphabetically** (Excellent=0, Fair=1, Good=2, Poor=3) would scramble the order, telling the model that "Poor" is the highest rating. For genuinely ordinal data the order is information you must not lose.

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#d97706;letter-spacing:1px">CHALLENGE 3 · CHOOSE A SCALER</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Resist the outlier</div>
<div style="color:#4a5578;margin-top:6px">Scale [20, 22, 21, 23, 22, 24, 1000] with StandardScaler, MinMaxScaler, and RobustScaler. Which keeps the six normal values spread out, and why?</div>
</div>

In [4]:
x = np.array([20, 22, 21, 23, 22, 24, 1000.0]).reshape(-1,1)
for name, sc in [("Standard",StandardScaler()),("MinMax",MinMaxScaler()),("Robust",RobustScaler())]:
    out = sc.fit_transform(x).ravel()
    print(f"{name:9}: inliers span {out[:-1].max()-out[:-1].min():.3f}, outlier at {out[-1]:.2f}")

Standard : inliers span 0.012, outlier at 2.45
MinMax   : inliers span 0.004, outlier at 1.00
Robust   : inliers span 2.000, outlier at 489.00


**Answer:** **RobustScaler** keeps the six inliers spread out; StandardScaler and especially MinMaxScaler crush them into a tiny range because the 1000 inflates the mean/SD and the max. RobustScaler centers on the **median** and scales by the **IQR**, both unmoved by a single extreme value (Chapter 21), so it is the scaler of choice when outliers are present. (Trees, by contrast, need no scaling at all.)

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#059669;letter-spacing:1px">CHALLENGE 4 · BIN A VARIABLE</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Width vs frequency</div>
<div style="color:#4a5578;margin-top:6px">Bin an income column into 4 groups two ways: equal-width with pd.cut and equal-frequency with pd.qcut. Show the counts per bin and explain the difference.</div>
</div>

In [5]:
income = pd.Series([18, 22, 25, 30, 35, 40, 55, 70, 90, 120, 200, 350])
width = pd.cut(income, bins=4)
freq  = pd.qcut(income, q=4)
print("equal-WIDTH (cut):"); print(width.value_counts().sort_index().to_string())
print("\nequal-FREQUENCY (qcut):"); print(freq.value_counts().sort_index().to_string())

equal-WIDTH (cut):
(17.668, 101.0]    9
(101.0, 184.0]     1
(184.0, 267.0]     1
(267.0, 350.0]     1

equal-FREQUENCY (qcut):
(17.999, 28.75]    3
(28.75, 47.5]      3
(47.5, 97.5]       3
(97.5, 350.0]      3


**Answer:** `pd.cut(bins=4)` makes **four equal-width** intervals over the range, so the high incomes pile most rows into the lowest bin and leave the top bins nearly empty. `pd.qcut(q=4)` makes **four equal-frequency** bins (quartiles), each with about the same count, but with very different widths. Use width when the scale is meaningful, frequency when you want balanced groups; both lose resolution, which is why modern models often skip binning.

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#2563eb;letter-spacing:1px">CHALLENGE 5 · BUILD A SAFE TRANSFORMER</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">No leakage</div>
<div style="color:#4a5578;margin-top:6px">Build a ColumnTransformer that scales numeric columns and one-hot encodes a categorical column, fit it on a training split, and transform the test split. Why fit on train only?</div>
</div>

In [6]:
df = pd.DataFrame({"age": rng.integers(20,60,120), "score": rng.normal(70,10,120), "plan":rng.choice(["A","B","C"],120)})
tr, te = train_test_split(df, test_size=0.3, random_state=1)
ct = ColumnTransformer([
    ("num", StandardScaler(), ["age","score"]),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["plan"]),
])
ct.fit(tr)
print("features:", list(ct.get_feature_names_out()))
print(f"train {ct.transform(tr).shape}, test {ct.transform(te).shape}")

features: ['num__age', 'num__score', 'cat__plan_A', 'cat__plan_B', 'cat__plan_C']
train (84, 5), test (36, 5)


**Answer:** The `ColumnTransformer` applies `StandardScaler` to the numeric columns and `OneHotEncoder` to the categorical one in a single object, and `get_feature_names_out()` shows the combined output columns. Fitting on the **training split only** means the scaler's mean/SD and the encoder's category list come from training data, so no information about the test set leaks into the transform. `handle_unknown="ignore"` keeps the test transform working if a category was unseen in training. Wrap the whole thing in a `Pipeline` with a model and the rule holds automatically inside cross-validation.

---
<div style="background:#ffffff;border:1px solid #e6e9f2;border-radius:16px;padding:22px 26px;font-family:Inter,sans-serif">
<div style="font-size:19px;font-weight:800;color:#1a2138">🎉 Nicely done!</div>
<div style="color:#4a5578;line-height:1.8;margin-top:8px">You encoded nominal and ordinal categories correctly, chose an outlier-resistant scaler, binned a variable two ways, and built a leakage-safe transformer. Good features, fit honestly on training data, are what every model downstream is built on.</div>
</div>

<div style="text-align:center;color:#8b94b3;font-size:12px;margin-top:16px">Statistics, Data Science and AI: A Visual Handbook · © 2026 John Fisher</div>